# PySpark Kafka Streaming with DataFrame API

This notebook demonstrates step-by-step execution of a PySpark Streaming application consuming data from Kafka using Spark's DataFrame API.

## Step 1: Import Required Libraries

In [53]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, window, sum as _sum, avg, count, 
    max as _max, min as _min, current_timestamp, 
    to_timestamp, expr, lit, when, round as _round
)
from pyspark.sql.types import (
    StructType, StructField, StringType, 
    IntegerType, DoubleType, TimestampType
)

## Step 2: Define Configuration

In [54]:
BOOTSTRAP_SERVERS = "bootstrap.simple-kafka-cluster.us-central1.managedkafka.lateral-layout-474104-p8.cloud.goog:9092"
KAFKA_TOPIC = "orders"
CONSUMER_GROUP = "data228.consumer.group"
CURRENT_DIR = os.getcwd()
AUTH_JAR = os.path.join(CURRENT_DIR, "google-cloud-kafka-pyspark-auth-1.0.0.jar")

## Step 3: Define Data Schema

In [55]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("timestamp", StringType(), True)
])

## Step 4: Create Spark Session

In [56]:
spark = SparkSession.builder \
    .appName("KafkaOrderStreamConsumer_Notebook") \
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7,org.apache.kafka:kafka-clients:3.7.2") \
    .config("spark.jars", AUTH_JAR) \
    .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

25/10/04 09:06:15 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Step 5: Connect to Kafka Stream

In [57]:
# Configure OAuth authentication for Google Cloud Managed Kafka
jaas_config = 'org.apache.kafka.common.security.oauthbearer.OAuthBearerLoginModule required;'

kafka_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("kafka.security.protocol", "SASL_SSL") \
    .option("kafka.sasl.mechanism", "OAUTHBEARER") \
    .option("kafka.sasl.jaas.config", jaas_config) \
    .option("kafka.sasl.login.callback.handler.class", 
            "com.google.cloud.hosted.kafka.auth.GcpLoginCallbackHandler") \
    .option("kafka.group.id", CONSUMER_GROUP) \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

kafka_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



25/10/04 09:06:18 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and/or set the option 'kafka.session.timeout.ms' to be very small so that the Kafka
 consumers from the previous query are marked dead by the Kafka group coordinator before the
 restarted query starts running.
    


## Step 6: Parse JSON Data

Transform raw Kafka data by:
- Converting value column from bytes to string
- Parsing JSON using the defined schema
- Flattening nested fields

In [60]:
# Parse JSON from Kafka value field
parsed_df = kafka_df.select(
    from_json(col("value").cast("string"), order_schema).alias("data"),
    col("timestamp").alias("kafka_timestamp")
).select(
    col("data.*"),
    col("kafka_timestamp")
)

orders_df = parsed_df.withColumn(
    "order_timestamp", 
    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS")
)

orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



## Step 7: Apply Transformations

Add calculated columns:
- Total amount per order
- Order size categorization
- Processing timestamp

In [61]:
# Calculate total amount
enriched_df = orders_df.withColumn(
    "total_amount", 
    _round(col("quantity") * col("price"), 2)
)

# Categorize order size based on quantity
enriched_df = enriched_df.withColumn(
    "order_size",
    when(col("quantity") <= 2, "Small")
    .when(col("quantity") <= 5, "Medium")
    .otherwise("Large")
)

enriched_df = enriched_df.withColumn(
    "processing_time",
    current_timestamp()
)

enriched_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- order_size: string (nullable = false)
 |-- processing_time: timestamp (nullable = false)



## Step 8: Apply Filtering Operations

In [62]:
# Filter: High-value orders (total_amount > 50)
high_value_orders = enriched_df.filter(col("total_amount") > 50)

# Filter: Large orders (quantity > 5)
large_orders = enriched_df.filter(col("quantity") > 5)

# Filter: Specific product
laptop_orders = enriched_df.filter(col("product") == "Laptop")

# Filter: Complex condition (Medium or Large orders with high value)
premium_orders = enriched_df.filter(
    (col("order_size").isin(["Medium", "Large"])) & 
    (col("total_amount") > 40)
)

## Step 9: Basic Aggregations by Product

In [63]:
product_stats = enriched_df.groupBy("product").agg(
    count("*").alias("total_orders"),
    _sum("quantity").alias("total_quantity"),
    avg("quantity").alias("avg_quantity"),
    _sum("total_amount").alias("total_revenue"),
    avg("price").alias("avg_price"),
    _min("price").alias("min_price"),
    _max("price").alias("max_price")
).select(
    col("product"),
    col("total_orders"),
    col("total_quantity"),
    _round(col("avg_quantity"), 2).alias("avg_quantity"),
    _round(col("total_revenue"), 2).alias("total_revenue"),
    _round(col("avg_price"), 2).alias("avg_price"),
    _round(col("min_price"), 2).alias("min_price"),
    _round(col("max_price"), 2).alias("max_price")
)

## Step 10: Window-Based Aggregations

Apply 30-second tumbling window with 1-minute watermark for handling late data.

In [64]:
windowed_stats = enriched_df \
    .withWatermark("order_timestamp", "1 minute") \
    .groupBy(
        window(col("order_timestamp"), "30 seconds"),
        col("product")
    ).agg(
        count("*").alias("orders_count"),
        _sum("quantity").alias("total_quantity"),
        _sum("total_amount").alias("revenue"),
        avg("total_amount").alias("avg_order_value")
    ).select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("product"),
        col("orders_count"),
        col("total_quantity"),
        _round(col("revenue"), 2).alias("revenue"),
        _round(col("avg_order_value"), 2).alias("avg_order_value")
    )

## Step 11: Multi-Level Aggregations

In [65]:
# Aggregate by product and order_size
product_size_stats = enriched_df.groupBy("product", "order_size").agg(
    count("*").alias("order_count"),
    _sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_revenue_per_order")
).select(
    col("product"),
    col("order_size"),
    col("order_count"),
    _round(col("total_revenue"), 2).alias("total_revenue"),
    _round(col("avg_revenue_per_order"), 2).alias("avg_revenue_per_order")
).orderBy(col("product"), col("order_size"))

## Step 12: Output Stream - Console Mode (Append)

In [66]:
query_console = enriched_df \
    .select("order_id", "product", "quantity", "price", "total_amount", "order_size", "order_timestamp") \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .option("numRows", 20) \
    .start()

25/10/04 09:07:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-c18b2368-e88c-450d-b389-e3f6c982c08a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/04 09:07:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/10/04 09:07:25 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-------+--------+-----+------------+----------+---------------+
|order_id|product|quantity|price|total_amount|order_size|order_timestamp|
+--------+-------+--------+-----+------------+----------+---------------+
+--------+-------+--------+-----+------------+----------+---------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+-------+--------+------+------------+----------+-------------------------+
|order_id   |product|quantity|price |total_amount|order_size|order_timestamp          |
+-----------+-------+--------+------+------------+----------+-------------------------+
|order-34730|printer|4       |741.88|2967.52     |Medium    |2025-10-04 09:07:36.53434|
+-----------+-------+--------+------+------------+----------+-------------------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+-------+--------+-----+------------+----------+--------------------------+
|order_id   |product|quantity|price|total_amount|order_size|order_timestamp           |
+-----------+-------+--------+-----+------------+----------+--------------------------+
|order-15884|printer|3       |489.9|1469.7      |Medium    |2025-10-04 09:07:37.134482|
+-----------+-------+--------+-----+------------+----------+--------------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+----------+--------+-------+------------+----------+--------------------------+
|order_id   |product   |quantity|price  |total_amount|order_size|order_timestamp           |
+-----------+----------+--------+-------+------------+----------+--------------------------+
|order-44088|headphones|10      |613.95 |6139.5      |Large     |2025-10-04 09:07:37.65626 |
|order-52620|printer   |6       |1542.19|9253.14     |Large     |2025-10-04 09:07:38.160887|
+-----------+----------+--------+-------+------------+----------+--------------------------+

-------------------------------------------
Batch: 4
-------------------------------------------
+-----------+--------+--------+-------+------------+----------+--------------------------+
|order_id   |product |quantity|price  |total_amount|order_size|order_timestamp           |
+-----------+--------+--------+-------+------------+----------+--

## Step 13: Output Stream - Aggregation Mode (Complete)

In [38]:
query_aggregation = product_stats \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

25/10/04 08:59:31 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-ab565f7c-4a39-4320-ba0a-cab7dcecc89d. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/04 08:59:31 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/10/04 08:59:32 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and

-------------------------------------------
Batch: 12
-------------------------------------------
+-------------------+-------------------+----------+------------+--------------+-------+---------------+
|window_start       |window_end         |product   |orders_count|total_quantity|revenue|avg_order_value|
+-------------------+-------------------+----------+------------+--------------+-------+---------------+
|2025-10-04 08:56:30|2025-10-04 08:57:00|printer   |1           |5             |10981.9|10981.9        |
|2025-10-04 08:56:00|2025-10-04 08:56:30|mouse     |1           |2             |3979.18|3979.18        |
|2025-10-04 08:55:30|2025-10-04 08:56:00|mouse     |2           |2             |1566.65|783.33         |
|2025-10-04 08:56:30|2025-10-04 08:57:00|webcam    |1           |1             |1272.92|1272.92        |
|2025-10-04 08:56:30|2025-10-04 08:57:00|laptop    |1           |4             |6032.92|6032.92        |
|2025-10-04 08:55:30|2025-10-04 08:56:00|webcam    |1         

-------------------------------------------
Batch: 12
-------------------------------------------
+-----------+----------+--------+-------+------------+--------------------------+
|order_id   |product   |quantity|price  |total_amount|order_timestamp           |
+-----------+----------+--------+-------+------------+--------------------------+
|order-97584|mouse     |1       |245.22 |245.22      |2025-10-04 08:55:50.198524|
|order-16442|mouse     |1       |1321.43|1321.43     |2025-10-04 08:55:55.203912|
|order-86409|printer   |1       |1268.08|1268.08     |2025-10-04 08:56:00.208299|
|order-56878|speaker   |3       |1545.04|4635.12     |2025-10-04 08:56:15.226752|
|order-90674|mouse     |2       |1989.59|3979.18     |2025-10-04 08:56:20.232745|
|order-47070|tablet    |3       |967.77 |2903.31     |2025-10-04 08:56:25.238128|
|order-39950|headphones|1       |2136.61|2136.61     |2025-10-04 08:56:40.286992|
|order-66353|speaker   |1       |379.07 |379.07      |2025-10-04 08:56:05.213505|


[Stage 131:>(7 + 2) / 200][Stage 133:>(0 + 0) / 200][Stage 134:>  (0 + 0) / 3]

## Step 14: Output Stream - Window Aggregation (Update)

In [67]:
query_window = windowed_stats \
    .writeStream \
    .outputMode("update") \
    .format("console") \
    .option("truncate", False) \
    .start()

25/10/04 09:07:50 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-022622b8-dadf-431d-9200-dac8224b1cb5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/04 09:07:50 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/10/04 09:07:50 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+-------+--------+------+------------+----------+--------------------------+
|order_id   |product|quantity|price |total_amount|order_size|order_timestamp           |
+-----------+-------+--------+------+------------+----------+--------------------------+
|order-63694|phone  |8       |845.37|6762.96     |Large     |2025-10-04 09:07:49.812369|
+-----------+-------+--------+------+------------+----------+--------------------------+

-------------------------------------------
Batch: 20
-------------------------------------------
+-----------+--------+--------+-------+------------+----------+--------------------------+
|order_id   |product |quantity|price  |total_amount|order_size|order_timestamp           |
+-----------+--------+--------+-------+------------+----------+--------------------------+
|order-30755|keyboard|1       |1005.84|1005.84     |Small     |2025-10-04 09:07:50.31

[Stage 22:======>        (81 + 2) / 200][Stage 23:>                 (0 + 0) / 1]

## Step 15: Output Stream - Filtered Data (Append)

In [44]:
query_filtered = high_value_orders \
    .select("order_id", "product", "quantity", "price", "total_amount", "order_timestamp") \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", False) \
    .start()

25/10/04 09:01:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-9055489c-3c19-434c-8fac-7781de53f8d5. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/04 09:01:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/10/04 09:01:02 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and

-------------------------------------------
Batch: 1
-------------------------------------------
+----------+------------+--------------+------------+-------------+---------+---------+---------+
|product   |total_orders|total_quantity|avg_quantity|total_revenue|avg_price|min_price|max_price|
+----------+------------+--------------+------------+-------------+---------+---------+---------+
|tablet    |1           |3             |3.0         |2903.31      |967.77   |967.77   |967.77   |
|laptop    |1           |4             |4.0         |6032.92      |1508.23  |1508.23  |1508.23  |
|speaker   |1           |3             |3.0         |4635.12      |1545.04  |1545.04  |1545.04  |
|mouse     |1           |2             |2.0         |3979.18      |1989.59  |1989.59  |1989.59  |
|printer   |1           |5             |5.0         |10981.9      |2196.38  |2196.38  |2196.38  |
|webcam    |1           |1             |1.0         |1272.92      |1272.92  |1272.92  |1272.92  |
|headphones|1        

-------------------------------------------
Batch: 6
-------------------------------------------
+-------------------+-------------------+----------+------------+--------------+-------+---------------+
|window_start       |window_end         |product   |orders_count|total_quantity|revenue|avg_order_value|
+-------------------+-------------------+----------+------------+--------------+-------+---------------+
|2025-10-04 08:56:30|2025-10-04 08:57:00|printer   |1           |5             |10981.9|10981.9        |
|2025-10-04 08:56:30|2025-10-04 08:57:00|webcam    |1           |1             |1272.92|1272.92        |
|2025-10-04 08:56:30|2025-10-04 08:57:00|headphones|1           |1             |2136.61|2136.61        |
+-------------------+-------------------+----------+------------+--------------+-------+---------------+



-------------------------------------------
Batch: 0
-------------------------------------------
+-------+------------+--------------+------------+-------------+---------+---------+---------+
|product|total_orders|total_quantity|avg_quantity|total_revenue|avg_price|min_price|max_price|
+-------+------------+--------------+------------+-------------+---------+---------+---------+
|monitor|1           |10            |10.0        |22335.9      |2233.59  |2233.59  |2233.59  |
+-------+------------+--------------+------------+-------------+---------+---------+---------+



-------------------------------------------
Batch: 6
-------------------------------------------
+-----------+----------+--------+-------+------------+--------------------------+
|order_id   |product   |quantity|price  |total_amount|order_timestamp           |
+-----------+----------+--------+-------+------------+--------------------------+
|order-64701|headphones|1       |2355.23|2355.23     |2025-10-04 08:59:24.196277|
|order-65806|phone     |5       |1717.89|8589.45     |2025-10-04 08:59:28.486427|
|order-50939|phone     |4       |206.61 |826.44      |2025-10-04 08:59:30.002016|
|order-72090|speaker   |7       |2422.85|16959.95    |2025-10-04 08:59:31.517921|
|order-71050|printer   |9       |1851.32|16661.88    |2025-10-04 08:59:32.024509|
|order-36002|speaker   |10      |487.94 |4879.4      |2025-10-04 08:59:32.529847|
|order-10420|monitor   |10      |2233.59|22335.9     |2025-10-04 08:59:33.041996|
|order-55412|webcam    |5       |432.59 |2162.95     |2025-10-04 08:59:23.482192|
|

[Stage 142:(38 + 2) / 200][Stage 144:>  (0 + 0) / 3][Stage 145:>  (0 + 0) / 1]

## Step 16: Monitor Active Queries

In [45]:
for query in spark.streams.active:
    print(f"Query ID: {query.id}")
    print(f"Status: {query.status}")
    print("-" * 60)

Query ID: 436d08e5-b757-412f-a8fa-bc000441fa74
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: 9cfb41a8-ecfc-41e1-96cc-c1cafe155beb
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: ea39e9e5-bf89-4923-bc9d-65e6ff805aa5
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: 0dc020af-77f5-474a-9def-b24e32f28550
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: 9118a9c1-aea9-473b-b7a5-e65f8b247f65
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
--------------------------------------------------------

[Stage 142:(66 + 2) / 200][Stage 144:>  (0 + 0) / 3][Stage 145:>  (0 + 0) / 1]

## Step 17: Stop Individual Queries

In [46]:
# Uncomment to stop specific queries
query_console.stop()
query_aggregation.stop()
query_window.stop()
query_filtered.stop()

25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 5, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 1, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 0, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
25/10/04 09:02:44 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 0, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.
25/10/04 09:02:44 ERRO

## Step 18: Advanced Aggregation with Custom Metrics

In [47]:
advanced_stats = enriched_df.groupBy("product").agg(
    count("*").alias("total_orders"),
    _sum("quantity").alias("total_units_sold"),
    _sum("total_amount").alias("total_revenue"),
    avg("total_amount").alias("avg_order_value"),
    _max("total_amount").alias("max_order_value"),
    _min("total_amount").alias("min_order_value")
).select(
    col("product"),
    col("total_orders"),
    col("total_units_sold"),
    _round(col("total_revenue"), 2).alias("total_revenue"),
    _round(col("avg_order_value"), 2).alias("avg_order_value"),
    _round(col("max_order_value"), 2).alias("max_order_value"),
    _round(col("min_order_value"), 2).alias("min_order_value")
).withColumn(
    "revenue_per_unit",
    _round(col("total_revenue") / col("total_units_sold"), 2)
)

query_advanced = advanced_stats \
    .writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", False) \
    .start()

25/10/04 09:02:46 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-2431f160-e0fc-4a7b-84b8-38427dd7b7d1. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/10/04 09:02:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/10/04 09:02:46 WARN KafkaSourceProvider: Kafka option 'kafka.group.id' has been set on this query, it is
 not recommended to set this option. This option is unsafe to use since multiple concurrent
 queries or sources using the same group id will interfere with each other as they are part
 of the same consumer group. Restarted queries may also suffer interference from the
 previous run having the same group id. The user should have only one query per group id,
 and

-------------------------------------------
Batch: 13
-------------------------------------------
+-------------------+-------------------+----------+------------+--------------+--------+---------------+
|window_start       |window_end         |product   |orders_count|total_quantity|revenue |avg_order_value|
+-------------------+-------------------+----------+------------+--------------+--------+---------------+
|2025-10-04 08:59:30|2025-10-04 09:00:00|speaker   |2           |17            |21839.35|10919.68       |
|2025-10-04 08:59:30|2025-10-04 09:00:00|phone     |2           |6             |1475.62 |737.81         |
|2025-10-04 08:59:30|2025-10-04 09:00:00|webcam    |1           |5             |994.0   |994.0          |
|2025-10-04 08:59:30|2025-10-04 09:00:00|tablet    |1           |4             |742.48  |742.48         |
|2025-10-04 08:59:00|2025-10-04 08:59:30|headphones|1           |1             |2355.23 |2355.23        |
|2025-10-04 08:59:30|2025-10-04 09:00:00|keyboard  |1 

-------------------------------------------
Batch: 13
-------------------------------------------
+-----------+----------+--------+-------+------------+--------------------------+
|order_id   |product   |quantity|price  |total_amount|order_timestamp           |
+-----------+----------+--------+-------+------------+--------------------------+
|order-64701|headphones|1       |2355.23|2355.23     |2025-10-04 08:59:24.196277|
|order-65806|phone     |5       |1717.89|8589.45     |2025-10-04 08:59:28.486427|
|order-50939|phone     |4       |206.61 |826.44      |2025-10-04 08:59:30.002016|
|order-72090|speaker   |7       |2422.85|16959.95    |2025-10-04 08:59:31.517921|
|order-71050|printer   |9       |1851.32|16661.88    |2025-10-04 08:59:32.024509|
|order-36002|speaker   |10      |487.94 |4879.4      |2025-10-04 08:59:32.529847|
|order-10420|monitor   |10      |2233.59|22335.9     |2025-10-04 08:59:33.041996|
|order-95618|monitor   |8       |1357.78|10862.24    |2025-10-04 08:59:37.266871|


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+--------+--------+-------+------------+--------------------------+
|order_id   |product |quantity|price  |total_amount|order_timestamp           |
+-----------+--------+--------+-------+------------+--------------------------+
|order-69544|keyboard|9       |2558.96|23030.64    |2025-10-04 08:59:46.757832|
+-----------+--------+--------+-------+------------+--------------------------+



25/10/04 09:03:11 WARN ExpiringCredentialRefreshingLogin: [Principal=:143442647069-compute@developer.gserviceaccount.com]: Expiring credential expires at 2025-10-04T09:08:43.307+0000, so buffer times of 60 and 300 seconds at the front and back, respectively, cannot be accommodated.  We will refresh at 2025-10-04T09:07:53.050+0000.
[Stage 146:(52 + 2) / 200][Stage 148:>  (0 + 0) / 1][Stage 150:>  (0 + 0) / 3]

## Step 19: Query Statistics and Performance Monitoring

In [48]:
for query in spark.streams.active:
    print(f"Query ID: {query.id}")
    print(f"Status: {query.status}")
    
    recent_progress = query.recentProgress
    if recent_progress:
        latest = recent_progress[-1]
        print(f"Batch ID: {latest.get('batchId', 'N/A')}")
        print(f"Input Rows: {latest.get('numInputRows', 0)}")
        print(f"Processing Rate: {latest.get('processedRowsPerSecond', 0)} rows/sec")
    print("-" * 60)

Query ID: 72654266-b602-411e-92df-a75606ae3587
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: 436d08e5-b757-412f-a8fa-bc000441fa74
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


[Stage 146:(53 + 2) / 200][Stage 148:>  (0 + 0) / 1][Stage 150:>  (0 + 0) / 3]

Batch ID: 5
Input Rows: 9
Processing Rate: 0.053918367591466516 rows/sec
------------------------------------------------------------
Query ID: 9cfb41a8-ecfc-41e1-96cc-c1cafe155beb
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID: 0
Input Rows: 1
Processing Rate: 0.0049256473532033945 rows/sec
------------------------------------------------------------
Query ID: ea39e9e5-bf89-4923-bc9d-65e6ff805aa5
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
------------------------------------------------------------
Query ID: 0dc020af-77f5-474a-9def-b24e32f28550
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID: 13
Input Rows: 27
Processing Rate: 0.12832577637094705 rows/sec
------------------------------------------------------------
Query ID: 422944cb-1d46-4f6c-aca8-de23edac96cb
Status: {'message': 'Processing new data', 'isDataAvailable': True, 

[Stage 146:(55 + 2) / 200][Stage 148:>  (0 + 0) / 1][Stage 150:>  (0 + 0) / 3]

Batch ID: 13
Input Rows: 23
Processing Rate: 0.10888501742160278 rows/sec
------------------------------------------------------------
Query ID: fde7cbc4-1fd6-40bd-9557-c019b4826543
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID: 11
Input Rows: 12
Processing Rate: 0.07566776805306834 rows/sec
------------------------------------------------------------
Query ID: 39eac5de-432e-4e6a-a774-7e10da6b1d22
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}


[Stage 146:(56 + 2) / 200][Stage 148:>  (0 + 0) / 1][Stage 150:>  (0 + 0) / 3]

Batch ID: 6
Input Rows: 26
Processing Rate: 0.14625393057438418 rows/sec
------------------------------------------------------------
Query ID: 628563bd-0440-4894-badd-f23d85ff5732
Status: {'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
Batch ID: 6
Input Rows: 3
Processing Rate: 0.01706863297318518 rows/sec
------------------------------------------------------------


[Stage 146:(72 + 2) / 200][Stage 148:>  (0 + 0) / 1][Stage 150:>  (0 + 0) / 3]

## Step 20: Stop All Queries and Cleanup

In [74]:
for query in spark.streams.active:
    query.stop()

spark.stop()

25/10/04 09:09:51 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thread.

## Summary

This notebook performed :

### DataFrame API Operations
1. **Transformations**: select, withColumn, when, cast, alias
2. **Filtering**: Simple and complex filter conditions
3. **Aggregations**: count, sum, avg, min, max
4. **Window Operations**: Time-based windowing with watermarks
5. **GroupBy**: Single and multi-dimensional grouping

### Streaming Concepts
- **Output Modes**: append, update, complete
- **Watermarking**: Handling late data
- **Kafka Integration**: OAuth authentication, schema parsing

### References
- [PySpark DataFrame API](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.html)
- [PySpark SQL Cheat Sheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf)
- [Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html)